In [1]:
import os

# Set the environment variable BEFORE importing modules that might use protobuf
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

In [2]:
import pandas as pd
import numpy as np  
import time
from datetime import datetime, timedelta
import talib
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from statsmodels.tsa.arima.model import ARIMA
import warnings
from xgboost import plot_importance
import seaborn as sns
import shap
import calendar
from datetime import date, timedelta

#import pandas_market_calendars as mcal

In [3]:
start_dt = '2001-01-01'

In [4]:
# --- 2. DATA LOADING AND PREPROCESSING (Same as before, but calculating direction) ---
base = r"D:\Work\wsl\jupyter\TimeSeriesDataAnalysis\corr-heat-map"

# Load DataFrames

hdfc_full_df = pd.read_csv(f"{base}/data-collect/HDFCBANK_NS_25y.csv", parse_dates=['Date'], index_col='Date').asfreq('B').ffill()

hdfc_full_df.drop(columns=["Dividends", "Stock Splits", 'Volume'], inplace=True, errors="ignore")

In [5]:
hdfc_full_df

,Open,High,Low,Close
Date,,,,
2000-08-11,19.253934,20.019833,19.253934,19.839621
2000-08-14,19.831429,20.028024,19.761802,19.946110
2000-08-15,19.946110,19.946110,19.946110,19.946110
2000-08-16,20.150894,20.273765,19.987065,20.150894
2000-08-17,20.044410,20.888128,20.044410,20.228718
...,...,...,...,...
2025-08-05,1986.500000,1994.400024,1965.500000,1977.599976
2025-08-06,1971.099976,1988.000000,1970.000000,1985.300049
2025-08-07,1980.099976,2001.599976,1976.800049,1995.400024


In [6]:
def calc_rsi(df: pd.DataFrame,
             period: int = 14) -> None:
    
    lags = [1]

    # Compute raw RSI once
    rsi_raw = talib.RSI(df["Close"].astype(float).values, timeperiod=period)
    rsi = pd.Series(rsi_raw / 100.0, index=df.index, name='RSI')

    # optional: store the level itself
    df['RSI'] = rsi
    
    df['RSI_Overbought'] = (rsi > 0.7).astype(int)  # Traditional 70 level
    df['RSI_Oversold'] = (rsi < 0.3).astype(int)    # Traditional 30 level
 
    df['RSI_Strong_Overbought'] = (rsi > 0.8).astype(int)  # 80 level for strong trends
    df['RSI_Strong_Oversold'] = (rsi < 0.2).astype(int)    # 20 level for 

    df['RSI_Momentum'] = rsi.diff()  # Directional change
    df['RSI_Acceleration'] = rsi.diff().diff()  # Rate of change of momentum

    df['RSI_Cross_50'] = ((rsi > 0.5) & (rsi.shift(1) <= 0.5)).astype(int)  # Bullish cross
    df['RSI_Cross_50_Bearish'] = ((rsi < 0.5) & (rsi.shift(1) >= 0.5)).astype(int)  # Bearish cross
    
    # 5. RSI divergence detection (simplified)
    price_high = df['High'].rolling(5).max()
    rsi_high = rsi.rolling(5).max()
    df['RSI_Bearish_Divergence'] = ((price_high > price_high.shift(5)) & 
                                   (rsi_high < rsi_high.shift(5))).astype(int)
    
    price_low = df['Low'].rolling(5).min()
    rsi_low = rsi.rolling(5).min()
    df['RSI_Bullish_Divergence'] = ((price_low < price_low.shift(5)) & 
                                   (rsi_low > rsi_low.shift(5))).astype(int)

    df['RSI_Volatility'] = rsi.rolling(20).std()  # RSI volatility
    df['RSI_ZScore'] = (rsi - rsi.rolling(20).mean()) / rsi.rolling(20).std()  # Standardized RSI

    # 10. RSI regime detection
    df['RSI_Trend'] = np.where(rsi > 0.5, 1, 0)  # Basic trend direction
    df['RSI_Trend_Strength'] = abs(rsi - 0.5) * 2  # Strength of trend (0-1 scale)
    
    features = ['RSI', 'RSI_Overbought', 'RSI_Oversold', 'RSI_Strong_Overbought', 'RSI_Strong_Oversold', 'RSI_Momentum', 'RSI_Acceleration', 'RSI_Cross_50', 'RSI_Cross_50_Bearish', 'RSI_Bearish_Divergence', 'RSI_Bullish_Divergence', 'RSI_Volatility', 'RSI_ZScore', 'RSI_Trend', 'RSI_Trend_Strength']

    # for feature in features:
    #     for n in lags:
    #         df[f"{feature}_lag_{n}"] = df[feature].shift(n)
 
calc_rsi(hdfc_full_df)

In [7]:
hdfc_full_df['Range'] = hdfc_full_df['High'] - hdfc_full_df['Low']  # Intraday volatility
hdfc_full_df['Momentum'] = hdfc_full_df['Close'] - hdfc_full_df['Open']  # Daily momentum
hdfc_full_df['Gap'] = hdfc_full_df['Open'] - hdfc_full_df['Close'].shift(1)  # Overnight gap (using prior close)

In [8]:
hdfc_full_df['Target'] = hdfc_full_df['Close'].pct_change().shift(-1)
data = hdfc_full_df.dropna(subset=['Target'])[start_dt:].copy() 


In [9]:
# Key points:
# Time Index: Create a integer-based time index that increments by 1 for each business day.
# Group IDs: Since you're dealing with a single stock (HDFC), all rows will belong to the same group (e.g., a constant group identifier).
# Features: Utilize OHLC data from; t−1 and derive additional features (e.g., volatility, momentum, gaps) to capture temporal patterns.
# Target: y (percentage change) is the variable to predict.


In [10]:
 
data['time_idx'] = (data.index - data.index[0]).days

# Create group ID (constant for all rows)
data['Group'] = "HDFC"

In [11]:
data.head(20)

,Open,High,Low,Close,RSI,RSI_Overbought,RSI_Oversold,RSI_Strong_Overbought,RSI_Strong_Oversold,RSI_Momentum,...,RSI_Volatility,RSI_ZScore,RSI_Trend,RSI_Trend_Strength,Range,Momentum,Gap,Target,time_idx,Group
Date,,,,,,,,,,,,,,,,,,,,,
2001-01-01,18.422505,18.492133,18.184955,18.205433,0.499576,0,0,0,0,-0.007088,...,0.061022,0.928917,0,0.000848,0.307178,-0.217072,0.172015,0.007874,0,HDFC
2001-01-02,18.254580,18.430696,17.939211,18.348782,0.522466,0,0,0,0,0.022890,...,0.055145,1.252303,1,0.044931,0.491486,0.094201,0.049148,0.008259,1,HDFC
2001-01-03,18.594523,19.249836,18.430694,18.500320,0.546102,0,0,0,0,0.023636,...,0.057420,1.488500,1,0.092203,0.819142,-0.094203,0.245742,-0.002214,2,HDFC
2001-01-04,18.705106,18.832073,18.402024,18.459364,0.538346,0,0,0,0,-0.007756,...,0.057086,1.227732,1,0.076692,0.430049,-0.245742,0.204785,0.014422,3,HDFC
2001-01-05,18.467560,18.840269,18.406123,18.725590,0.580091,0,0,0,0,0.041745,...,0.059838,1.717829,1,0.160183,0.434146,0.258030,0.008196,0.006780,4,HDFC
2001-01-08,18.594526,19.004097,18.594526,18.852554,0.598727,0,0,0,0,0.018636,...,0.065241,1.746651,1,0.197454,0.409571,0.258029,-0.131064,0.024332,7,HDFC
2001-01-09,18.897608,19.495581,18.774737,19.311275,0.657816,0,0,0,0,0.059089,...,0.075537,2.201929,1,0.315632,0.720844,0.413668,0.045054,-0.043903,8,HDFC
2001-01-10,19.544728,19.655312,18.356972,18.463461,0.508716,0,0,0,0,-0.149100,...,0.072565,0.288645,1,0.017432,1.298341,-1.081267,0.233453,0.015528,9,HDFC
2001-01-11,18.676437,19.004095,18.512610,18.750160,0.546176,0,0,0,0,0.037460,...,0.073392,0.778461,1,0.092353,0.491486,0.073723,0.212976,0.036697,10,HDFC


In [12]:
data.describe()

,Open,High,Low,Close,RSI,RSI_Overbought,RSI_Oversold,RSI_Strong_Overbought,RSI_Strong_Oversold,RSI_Momentum,...,RSI_Bullish_Divergence,RSI_Volatility,RSI_ZScore,RSI_Trend,RSI_Trend_Strength,Range,Momentum,Gap,Target,time_idx
count,6420.000000,6420.000000,6420.000000,6420.000000,6420.000000,6420.000000,6420.000000,6420.000000,6420.000000,6420.000000,...,6420.000000,6420.000000,6420.000000,6420.000000,6420.000000,6420.000000,6420.000000,6420.000000,6420.000000,6420.000000
mean,562.748941,568.010225,557.233748,562.732117,0.538584,0.084891,0.018536,0.007944,0.001713,-0.000009,...,0.066044,0.070149,0.004989,0.630062,0.197572,10.776477,-0.016825,0.321443,0.000892,4492.500000
std,572.838841,577.507676,568.191777,572.866654,0.115941,0.278741,0.134889,0.088781,0.041361,0.047160,...,0.248378,0.024141,1.218781,0.482825,0.143822,12.357767,9.625678,6.762727,0.017953,2594.813804
min,15.793470,16.031626,15.207343,15.709907,0.134540,0.000000,0.000000,0.000000,0.000000,-0.306190,...,0.000000,0.018905,-3.538147,0.000000,0.000018,0.000000,-102.644464,-106.507896,-0.206437,0.000000
25%,83.315620,85.068589,81.517340,83.170074,0.458664,0.000000,0.000000,0.000000,0.000000,-0.027412,...,0.000000,0.052282,-0.930171,0.000000,0.080295,2.524344,-2.136449,-0.737668,-0.007382,2246.250000
50%,293.710962,296.518464,288.900890,293.656387,0.539100,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.066541,-0.011327,1.000000,0.169216,6.316161,-0.012612,0.097424,0.000000,4492.500000
75%,1006.893118,1014.418797,997.173423,1005.417068,0.619554,0.000000,0.000000,0.000000,0.000000,0.026742,...,0.000000,0.085016,0.925915,1.000000,0.287920,15.319914,2.172370,1.418136,0.008699,6738.750000
max,2037.699951,2037.699951,2017.000000,2025.800049,0.868470,1.000000,1.000000,1.000000,1.000000,0.309843,...,1.000000,0.192241,3.265864,1.000000,0.736939,154.061612,86.977176,70.569550,0.258174,8985.000000


In [13]:
data.isna().sum()

Open                      0
High                      0
Low                       0
Close                     0
RSI                       0
RSI_Overbought            0
RSI_Oversold              0
RSI_Strong_Overbought     0
RSI_Strong_Oversold       0
RSI_Momentum              0
RSI_Acceleration          0
RSI_Cross_50              0
RSI_Cross_50_Bearish      0
RSI_Bearish_Divergence    0
RSI_Bullish_Divergence    0
RSI_Volatility            0
RSI_ZScore                0
RSI_Trend                 0
RSI_Trend_Strength        0
Range                     0
Momentum                  0
Gap                       0
Target                    0
time_idx                  0
Group                     0
dtype: int64

In [14]:
data[data['Target']> 100].head()

,Open,High,Low,Close,RSI,RSI_Overbought,RSI_Oversold,RSI_Strong_Overbought,RSI_Strong_Oversold,RSI_Momentum,...,RSI_Volatility,RSI_ZScore,RSI_Trend,RSI_Trend_Strength,Range,Momentum,Gap,Target,time_idx,Group
Date,,,,,,,,,,,,,,,,,,,,,


In [15]:
data[(data['Target'] == np.inf) | (data['Target'] == -np.inf) ].head()

,Open,High,Low,Close,RSI,RSI_Overbought,RSI_Oversold,RSI_Strong_Overbought,RSI_Strong_Oversold,RSI_Momentum,...,RSI_Volatility,RSI_ZScore,RSI_Trend,RSI_Trend_Strength,Range,Momentum,Gap,Target,time_idx,Group
Date,,,,,,,,,,,,,,,,,,,,,


In [16]:

print(f"NaN values in Target: {data['Target'].isna().sum()}")

NaN values in Target: 0


In [17]:
import pytorch_forecasting
print("PyTorch-Forecasting version:", pytorch_forecasting.__version__)

d:\Work\miniconda_env\hf-pytorch\lib\site-packages\transformers\utils\hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


PyTorch-Forecasting version: 1.4.0


In [18]:
# Check for extremely small values that might cause numerical instability
print("Extremely small Target values:", len(data[abs(data['Target']) < 1e-10]))

# Check for values that might cause issues with softplus transformation
print("Negative values that might cause softplus issues:", len(data[data['Target'] < -10]))

Extremely small Target values: 373
Negative values that might cause softplus issues: 0


In [19]:
len(data)

6420

In [20]:
#data = data[abs(data['Target']) >= 1e-10]

In [21]:
len(data)

6420

In [22]:
data = data.sort_values('Date')

# Determine split points
n_total = len(data)
n_train = int(0.75 * n_total)
n_val = int(0.15 * n_total)

#==========================

# Chronological split
# train_data = data.iloc[:n_train]
# val_data = data.iloc[n_train:n_train + n_val]
# test_data = data.iloc[n_train + n_val:]

# Ensure splits respect time order
# assert train_data['time_idx'].max() < val_data['time_idx'].min()
# assert val_data['time_idx'].max() < test_data['time_idx'].min()

max_encoder_length = 60  # Use 60 days of historical context
max_prediction_length = 1   # Predict next day's return


# Adjusted splits to include encoder context [Evaluation phase for more, use above for production]
train_data = data.iloc[:n_train]   
val_data = data.iloc[n_train - max_encoder_length : n_train + n_val]  
test_data = data.iloc[n_train + n_val - max_encoder_length :]  

#==========================


print("len(train_data): ", len(train_data), "\ntrain_data.index.min:" ,train_data.index.min())
print("len(val_data): ", len(val_data), "\nval_data.index.min:" ,val_data.index.min())
print("len(test_data): ", len(test_data), "\ntest_data.index.min:" ,test_data.index.min())

len(train_data):  4815 
train_data.index.min: 2001-01-01 00:00:00
len(val_data):  1023 
val_data.index.min: 2019-03-25 00:00:00
len(test_data):  702 
test_data.index.min: 2022-12-01 00:00:00


In [23]:
# How many and where
print(train_data["Target"].isna().sum(), "NA")
print(np.isinf(train_data["Target"]).sum(), "inf")
print(train_data[train_data["Target"].isna() | np.isinf(train_data["Target"])])

0 NA
0 inf
Empty DataFrame
Columns: [Open, High, Low, Close, RSI, RSI_Overbought, RSI_Oversold, RSI_Strong_Overbought, RSI_Strong_Oversold, RSI_Momentum, RSI_Acceleration, RSI_Cross_50, RSI_Cross_50_Bearish, RSI_Bearish_Divergence, RSI_Bullish_Divergence, RSI_Volatility, RSI_ZScore, RSI_Trend, RSI_Trend_Strength, Range, Momentum, Gap, Target, time_idx, Group]
Index: []

[0 rows x 25 columns]


In [24]:
from pytorch_forecasting import GroupNormalizer, TimeSeriesDataSet


 
training = TimeSeriesDataSet(
    train_data,
    time_idx="time_idx", # Specifies the integer-based time index column that ensures proper temporal ordering. This should increase by +1 for each consecutive business day
    target="Target",
    group_ids=["Group"],
    max_encoder_length=max_encoder_length, # Uses 60 trading days (≈3 months) of historical data as context for predictions, capturing short-term patterns and trends
    max_prediction_length=max_prediction_length, #Predicts the next day's return (single-step forecasting), appropriate for daily stock prediction
    static_categoricals=["Group"], # "Group" identifies this as HDFC Bank data (constant categorical identifier) 
    static_reals=[], # Empty as no continuous static features (e.g., company fundamentals) are included.
    time_varying_known_reals=["time_idx"], #"time_idx" is known for future periods 
    time_varying_unknown_reals=[
        "Open", "High", "Low", "Close", 
        "Range", "Momentum", "Gap", "Target"
    ], # OHLC prices and derived features (Range, Momentum, Gap) that are only known historically. "Target" is included as it's the variable being predicted 
    add_relative_time_idx=True, # Adds a relative position index within each sequence, helping the model learn temporal patterns 
    add_target_scales=True, #Includes scaling parameters as static features, helping the model adapt to different value ranges
    add_encoder_length=True, #: Adds the actual encoder length as a feature (useful if using variable length encoding) 
    target_normalizer=GroupNormalizer(
        groups=["Group"], transformation=None
    ), # GroupNormalizer normalizes the target variable within each group ("HDFC") using softplus transformation, which handles near-zero values well for percentage changes 
    allow_missing_timesteps=True,
)
print("✅ Dataset created successfully.")
print("Sample:", next(iter(training.to_dataloader(batch_size=2)))[0].keys())

#==========================


# For model development and tuning, use predict=False to create a robust validation set with multiple samples. 
# This helps better evaluate generalization and prevents overfitting.
# For final testing before deployment, use predict=True to simulate how the model will perform in production when making future predictions from the latest data.
# validation = TimeSeriesDataSet.from_dataset(training, val_data, predict=True, stop_randomization=True)
# test = TimeSeriesDataSet.from_dataset(training, test_data, predict=True, stop_randomization=True)

# Use above for production
validation = TimeSeriesDataSet.from_dataset(training, val_data, predict=False, stop_randomization=True)
test = TimeSeriesDataSet.from_dataset(training, test_data, predict=False, stop_randomization=True)

#==========================



batch_size = 64
train_dataloader = training.to_dataloader(train=True, batch_size=batch_size)
val_dataloader = validation.to_dataloader(train=False, batch_size=batch_size)
test_dataloader = test.to_dataloader(train=False, batch_size=batch_size)

✅ Dataset created successfully.
Sample: dict_keys(['encoder_cat', 'encoder_cont', 'encoder_target', 'encoder_lengths', 'decoder_cat', 'decoder_cont', 'decoder_target', 'decoder_lengths', 'decoder_time_idx', 'groups', 'target_scale'])


In [ ]:
import optuna
import lightning as L
from lightning.pytorch.callbacks import EarlyStopping
import torch.nn as nn
from pytorch_forecasting.metrics import MAE, RMSE, QuantileLoss
import os
import shutil
import tempfile
from pytorch_forecasting import TemporalFusionTransformer

def objective(trial):
    # Create a temporary directory for this trial
    with tempfile.TemporaryDirectory() as temp_dir:
        # Suggest hyperparameters
        hidden_size = trial.suggest_categorical("hidden_size", [8, 16, 32, 64, 128, 256])
        lstm_layers = trial.suggest_int("lstm_layers", 1, 12)
        attention_head_size = trial.suggest_categorical("attention_head_size", [2, 4, 8, 12])
        dropout = trial.suggest_float("dropout", 0.0, 0.4)
        learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
        hidden_continuous_size = trial.suggest_categorical("hidden_continuous_size", [4, 8, 16, 32])
        
        # Create model with suggested hyperparameters
        tft = TemporalFusionTransformer.from_dataset(
            training,
            hidden_size=hidden_size,
            lstm_layers=lstm_layers,
            attention_head_size=attention_head_size,
            dropout=dropout,
            loss=QuantileLoss(quantiles=[0.05, 0.25, 0.5, 0.75, 0.95]),
            learning_rate=learning_rate,
            optimizer="adamw",
            output_size=5,
            hidden_continuous_size=hidden_continuous_size,
            max_encoder_length=60,
            logging_metrics=nn.ModuleList([MAE(), RMSE()]),
        )
        
        # Define callbacks (without LearningRateMonitor to avoid logger dependency)
        early_stop_callback = EarlyStopping(
            monitor="val_loss",
            patience=5,
            min_delta=0.001,
            mode="min",
            verbose=False,
        )
        
        # Configure trainer without any logger
        trainer = L.Trainer(
            max_epochs=300,
            accelerator="auto",
            devices="auto",
            enable_model_summary=False,
            gradient_clip_val=0.1,
            callbacks=[early_stop_callback],
            enable_checkpointing=False,
            logger=False,  # Completely disable logging
            log_every_n_steps=5,
            enable_progress_bar=False,
        )
        
        # Train the model
        trainer.fit(
            tft,
            train_dataloaders=train_dataloader,
            val_dataloaders=val_dataloader,
        )
        
        # Get validation loss
        val_loss = trainer.callback_metrics["val_loss"].item()
        
        return val_loss

# Clean up any existing logs before starting
for dir_name in ["optuna_logs", "final_logs", "lightning_logs"]:
    if os.path.exists(dir_name):
        shutil.rmtree(dir_name)

# Create Optuna study
study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42), #multivariate=True
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5),
)

# Run optimization
print("Starting hyperparameter optimization...")
study.optimize(objective, n_trials=75, timeout=10800, n_jobs= -1)  # Reduced to 10 trials for testing

# Print results
print("\nBest trial:")
trial = study.best_trial
print(f"  Validation Loss: {trial.value:.6f}")
print("  Best Hyperparameters:")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

# Train final model with best hyperparameters
best_params = study.best_trial.params

print("\nTraining final model with best hyperparameters...")
tft_final = TemporalFusionTransformer.from_dataset(
    training,
    hidden_size=best_params["hidden_size"],
    lstm_layers=best_params["lstm_layers"],
    attention_head_size=best_params["attention_head_size"],
    dropout=best_params["dropout"],
    loss=QuantileLoss(quantiles=[0.05, 0.25, 0.5, 0.75, 0.95]),
    learning_rate=best_params["learning_rate"],
    optimizer="adamw",
    output_size=5,
    hidden_continuous_size=best_params["hidden_continuous_size"],
    max_encoder_length=60,
    logging_metrics=nn.ModuleList([MAE(), RMSE()]),
)

# Final training with more epochs (still no logger to avoid issues)
trainer_final = L.Trainer(
    max_epochs=300,
    accelerator="auto",
    devices="auto",
    enable_model_summary=True,
    gradient_clip_val=0.1,
    callbacks=[EarlyStopping(monitor="val_loss", patience=10, mode="min")],
    enable_checkpointing=False,
    logger=False,  # No logger to avoid file issues
    log_every_n_steps=20,
)

trainer_final.fit(
    tft_final,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader,
)

 

[I 2025-08-30 16:30:09,483] A new study created in memory with name: no-name-39e5e216-1918-476c-8d03-0030848e92d1


Starting hyperparameter optimization...

d:\Work\miniconda_env\hf-pytorch\lib\site-packages\lightning\pytorch\utilities\parsing.py:209: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
d:\Work\miniconda_env\hf-pytorch\lib\site-packages\lightning\pytorch\utilities\parsing.py:209: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
GPU available: True (cuda), used: True
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
GPU available: True (cuda), used: True
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
GPU available: True (cuda), used: True
TPU available

In [ ]:
# After training, you can evaluate on test data
test_dataloader = test.to_dataloader(
    train=False, 
    batch_size=batch_size
)

test_metrics = trainer_final.test(tft_final, dataloaders=test_dataloader)
print(f"Test metrics: {test_metrics}")

In [ ]:
# optuna.visualization.plot_optimization_history

In [ ]:
import pandas as pd
import numpy as np

# Get predictions on test data
predictions = tft_final.predict(test_dataloader, return_y=True, mode="raw")

# Extract actual and predicted values
# For quantile regression, we typically use the median (0.5 quantile) as point forecast
y_actual = predictions.y[0].cpu().numpy().flatten()  # Actual values
y_pred_median = predictions.output[0][:, :, 1].cpu().numpy().flatten()  # Median prediction (0.5 quantile)

# Get dates from test data
# Assuming your test_data has a DateTime index
test_dates = test_data.index[-len(y_actual):]  # Align dates with predictions

# Create results DataFrame
results_df = pd.DataFrame({
    'Date': test_dates,
    'Actual_Return': y_actual,
    'Predicted_Return': y_pred_median,
    'Absolute_Error': np.abs(y_actual - y_pred_median)
})

# Calculate additional metrics
results_df['Prediction_Direction'] = np.sign(results_df['Predicted_Return'])
results_df['Actual_Direction'] = np.sign(results_df['Actual_Return'])
results_df['Direction_Correct'] = results_df['Prediction_Direction'] == results_df['Actual_Direction']

# Display first few rows
#print("Results DataFrame:")
#print(results_df.head(10))

# Calculate and display performance metrics
direction_accuracy = results_df['Direction_Correct'].mean() * 100
mean_absolute_error = results_df['Absolute_Error'].mean()

print(f"\nDirection Prediction Accuracy: {direction_accuracy:.2f}%")
print(f"Mean Absolute Error: {mean_absolute_error:.6f}")

# Save to CSV for further analysis
results_df.to_csv('tft_predictions_results.csv', index=False)
print("\nResults saved to 'tft_predictions_results.csv'")

In [ ]:
results_df.head()

In [ ]:
# TRAINING MODE (predict=False):
# Encoder: [t-60, t-59, ..., t-1] → Uses actual targets
# Decoder: [t, t+1, ..., t+n] → Uses actual targets (teacher forcing)

# INFERENCE MODE (predict=True):
# Encoder: [t-60, t-59, ..., t-1] → Uses actual targets  
# Decoder: [t, t+1, ..., t+n] → IGNORES your targets, uses model predictions

test_data_wo_target = hdfc_full_df.iloc[-60:].copy()
test_data_wo_target['time_idx'] = (test_data_wo_target.index - pd.Timestamp(start_dt)).days
test_data_wo_target['Group'] = "HDFC"
#Model generates its own predictions autoregressively
test_data_wo_target['Target'] = test_data_wo_target['Target'].fillna(0)
test_data_wo_target.head(70)

In [ ]:
test_data_wo_target = test_data_wo_target
inference_ds = TimeSeriesDataSet.from_dataset(
    training,                      # re-uses all encoders / scalers
    test_data_wo_target,
    predict=True,                  # <- predict mode
    stop_randomization=True
)

test_dataloader_wo_target = inference_ds.to_dataloader(train=False, batch_size=1, num_workers=0)


# Get predictions on inference_ds data
predictions = tft_final.predict(test_dataloader_wo_target, mode="quantiles")
print("predictions:", predictions)

predicted_return = predictions[0, -1, 2].item() 
# For risk management, consider the prediction intervals
lower_bound = predictions[0, -1, 0].item()  # -0.0065 (-0.65%)
upper_bound = predictions[0, -1, 4].item()  # 0.0193 (+1.93%)

print(f"Predicted return: {predicted_return:.4f} ({predicted_return*100:.2f}%)")
print(f"90% prediction interval: [{lower_bound:.4f}, {upper_bound:.4f}]")


In [ ]:
predictions = tft_final.predict(test_dataloader_wo_target, mode="prediction")
print("predictions:", predictions)
predicted_return = predictions[0, -1].item()  # Access first batch, last prediction step
print("predicted_return:", predicted_return)
